# Option B: Generate VLM Captions

This notebook implements the visual recaptioning pipeline. It processes all raw scientific figures, charts, and tables in the dataset, transcribing all textual details, columns, rows, labels, and trends into rich, searchable captions. These dense visual captions are then injected into our search index to unlock hybrid multimodal retrieval.

Current Model Options:
- [x] **Qwen/Qwen2-VL-7B-Instruct:** General-purpose visual parser and structural layout analyst.
- [] **meta-llama/Llama-3.2-11B-Vision (4-bit quantization):** High-fidelity data extraction from complex charts, multi-column PDFs, and sequential graphs.
- [] **microsoft/Phi-4-multimodal-instruct:** Blazing-fast, low-latency document reading and tabular data extraction.
- [] **OpenGVLab/InternVL2_5-26B (4-bit quantization):** Reading ultra-dense text, small-print footnotes, and technical schematics.

The cached image captions can be found in ../outputs/cache/vlm_image_captions

In [ ]:
import os
import ctypes

# Force CUDA library paths for NVRTC and bitsandbytes compatibility
cuda_path = "/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/nvidia/cu13/lib/"
if cuda_path not in os.environ.get("LD_LIBRARY_PATH", ""):
    os.environ["LD_LIBRARY_PATH"] = cuda_path + ":" + os.environ.get("LD_LIBRARY_PATH", "")

# Force CUDA linking prior to loading PyTorch/Transformers to guarantee bitsandbytes initializes correctly
try:
    ctypes.CDLL(os.path.join(cuda_path, "libnvJitLink.so.13"))
    ctypes.CDLL(os.path.join(cuda_path, "libnvrtc.so.13"))
    ctypes.CDLL(os.path.join(cuda_path, "libnvrtc-builtins.so.13.0"))
    print("✅ CUDA libraries pre-loaded successfully!")
except Exception as e:
    print(f"⚠️ Pre-loading CUDA libraries failed: {e}")

In [ ]:
import json
import torch
from pathlib import Path
from tqdm.notebook import tqdm
from PIL import Image
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

# Configure paths
base_dir = Path("..").resolve()
cache_dir = base_dir / "outputs/cache"
cache_dir.mkdir(parents=True, exist_ok=True)

save_path = cache_dir / "vlm_image_captions.json"
data_dir = base_dir / "data"

# 1. Gather all unique images referenced in the dataset
unique_images = set()
for fname in ["train.jsonl", "test.jsonl"]:
    fpath = data_dir / fname
    if not fpath.exists():
        continue
    with open(fpath, "r", encoding="utf-8") as f:
        for line in f:
            d = json.loads(line)
            for iq in d.get("img_quotes", []):
                unique_images.add(iq["img_path"])
                
total_images = len(unique_images)
print(f"📊 Total unique images referenced in dataset: {total_images}")

In [ ]:
# 2. Load existing progress for seamless resume capability
progress = {}
if save_path.exists():
    try:
        with open(save_path, "r", encoding="utf-8") as f:
            progress = json.load(f)
        print(f"🔄 Resuming: Found {len(progress)} images already captioned!")
    except Exception as e:
        print(f"⚠️ Failed to load existing cache, starting fresh: {e}")
        
# Determine which images still need processing
to_process = [img for img in unique_images if img not in progress]
print(f"⏳ Images remaining to process: {len(to_process)}")

In [ ]:
# 3. Load Qwen2-VL-7B-Instruct in 4-bit Precision
if to_process:
    print("🚀 Loading Qwen2-VL-7B-Instruct in 4-bit quantization...")
    from transformers import BitsAndBytesConfig
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    
    model_id = "Qwen/Qwen2-VL-7B-Instruct"
    
    # Limit pixel footprint to optimize VRAM and inference speed
    min_pixels = 256 * 28 * 28
    max_pixels = 1024 * 28 * 28 # Capping at 1024 preserves high OCR detail
    
    processor = AutoProcessor.from_pretrained(
        model_id, 
        min_pixels=min_pixels, 
        max_pixels=max_pixels
    )
    
    model = Qwen2VLForConditionalGeneration.from_pretrained(
        model_id,
        quantization_config=quant_config,
        device_map="auto",
        torch_dtype=torch.bfloat16
    )
    print("🔥 Model and Processor loaded successfully!")
else:
    print("✅ All images are already processed! No model loading required.")

In [ ]:
# 4. Main Inference Loop
if to_process:
    prompt_text = (
        "Describe this scientific figure, table, or chart in maximum detail. "
        "Transcribe all textual content, table values, rows, columns, labels, legends, and data trends inside the image "
        "explicitly so a researcher can perfectly understand the data without seeing the image. Avoid high-level summaries; yield detailed transcripts."
    )
    
    pbar = tqdm(to_process, desc="Visual Recaptioning Progress")
    save_interval = 25
    count_since_save = 0
    
    for img_rel_path in pbar:
        # Search directories (handles nested structures)
        search_paths = [
            base_dir / "data/images" / img_rel_path,
            base_dir / "data" / img_rel_path,
            base_dir / img_rel_path
        ]
        
        full_image_path = None
        for path_cand in search_paths:
            if path_cand.exists():
                full_image_path = path_cand
                break
                
        if not full_image_path:
            progress[img_rel_path] = "Error: Image file not found."
            continue
            
        try:
            messages = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "image": str(full_image_path)},
                        {"type": "text", "text": prompt_text},
                    ],
                }
            ]
            
            text_input = processor.apply_chat_template(
                messages, 
                tokenize=False, 
                add_generation_prompt=True
            )
            image_inputs, video_inputs = process_vision_info(messages)
            
            inputs = processor(
                text=[text_input],
                images=image_inputs,
                videos=video_inputs,
                padding=True,
                return_tensors="pt",
            )
            inputs = inputs.to("cuda")
            
            with torch.no_grad():
                generated_ids = model.generate(**inputs, max_new_tokens=350)
                
            generated_ids_trimmed = [
                out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
            ]
            output_text = processor.batch_decode(
                generated_ids_trimmed, 
                skip_special_tokens=True, 
                clean_up_tokenization_spaces=False
            )[0]
            
            progress[img_rel_path] = output_text.strip()
            
        except Exception as e:
            progress[img_rel_path] = f"Error: {str(e)}"
            
        count_since_save += 1
        if count_since_save >= save_interval:
            with open(save_path, "w", encoding="utf-8") as f:
                json.dump(progress, f, indent=2, ensure_ascii=False)
            count_since_save = 0
            
        pbar.set_postfix({"processed": f"{len(progress)}/{total_images}"})

    # Final Save
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(progress, f, indent=2, ensure_ascii=False)
    print(f"\n🎉 SUCCESS: Recaptioning finished! Captions saved to: {save_path}")
else:
    print("✅ No images to process.")